In [1]:
!git clone https://github.com/Uyen-nt/MTG-downstreamtask.git

Cloning into 'MTG-downstreamtask'...
remote: Enumerating objects: 3404, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 3404 (delta 134), reused 38 (delta 38), pack-reused 3235 (from 4)
Receiving objects: 100% (3404/3404), 1.22 GiB | 44.31 MiB/s, done.
Resolving deltas: 100% (2047/2047), done.
Updating files: 100% (297/297), done.


In [2]:
%cd MTG-downstreamtask

/kaggle/working/MTG-downstreamtask


In [ ]:
!git pull origin main

In [3]:
!pip install pytorch-pretrained-bert

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 24.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!git pull origin main

In [ ]:
#!python behrt/train_synthetic.py

In [5]:
import torch
from torch.utils.data import DataLoader
from behrt.model import BertForMultiLabelPrediction, BertConfig
from behrt.data_mimic_loader import RealBEHRTDataset
from behrt.optimizer import adam
import numpy as np

# ============================
# 1) LOAD MIMIC REAL DATA
# ============================

train_data = RealBEHRTDataset("/kaggle/input/behrt-ft-cp/behrt_compare/mimic_train.npz", max_len=512)
val_data   = RealBEHRTDataset("/kaggle/input/behrt-ft-cp/behrt_compare/mimic_val.npz",   max_len=512)
test_data  = RealBEHRTDataset("/kaggle/input/behrt-ft-cp/behrt_compare/mimic_test.npz",  max_len=512)

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=8, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=8, shuffle=False)

print("✔ Loaded mimic fine-tune datasets")

# ============================
# 2) LOAD MODEL PRETRAINED ON SYNTHETIC
# ============================
config = {
    'vocab_size': 2871,
    'hidden_size': 288,
    'seg_vocab_size': 34,
    'age_vocab_size': 2,
    'max_position_embedding': 1500,
    'hidden_dropout_prob': 0.1,
    'num_hidden_layers': 4,
    'num_attention_heads': 8,
    'attention_probs_dropout_prob': 0.1,
    'intermediate_size': 512,
    'hidden_act': 'gelu',
    'initializer_range': 0.02,
}
feature_dict = {
    'word':True,
    'seg':True,
    'age':False,
    'position': True
}

model_config = BertConfig(config)
model = BertForMultiLabelPrediction(model_config, num_labels=config['vocab_size'], feature_dict=feature_dict)

print("🔹 Loading pretrained weights from synthetic ...")
model.load_state_dict(torch.load("/kaggle/input/behrt-ft-cp/behrt_compare/best_val_model.pt", map_location="cpu"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("✔ Model loaded on", device)

# ============================
# 3) FREEZE LOWER LAYERS
# ============================
for name, param in model.named_parameters():
    if ("embeddings" in name) or ("encoder.layer.0" in name) or ("encoder.layer.1" in name):
        param.requires_grad = False
        # print("❄ FREEZE", name)
    else:
        param.requires_grad = True

print("✔ Finished freezing early layers")

# ============================
# 4) SET OPTIMIZER — ONLY TRAIN HOT LAYERS
# ============================
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = adam(model.named_parameters())

# ============================
# 5) TRAIN + VAL
# ============================
best_val_loss = float('inf')
patience = 5
wait = 0

for epoch in range(20):
    # ===== TRAIN =====
    model.train()
    total_loss = 0
    for batch in train_loader:
        visits, age_ids, seg_ids, pos_ids, attention_mask, labels = batch
        visits, age_ids, seg_ids, pos_ids, attention_mask, labels = \
            visits.to(device), age_ids.to(device), seg_ids.to(device), pos_ids.to(device), attention_mask.to(device), labels.to(device)

        loss, logits = model(visits, age_ids, seg_ids, pos_ids, attention_mask, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"\n🟩 Fine-tune Epoch {epoch} TRAIN loss = {avg_train_loss:.4f}")

    # ===== VALIDATE =====
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            visits, age_ids, seg_ids, pos_ids, attention_mask, labels = batch
            visits, age_ids, seg_ids, pos_ids, attention_mask, labels = \
                visits.to(device), age_ids.to(device), seg_ids.to(device), pos_ids.to(device), attention_mask.to(device), labels.to(device)

            loss, _ = model(visits, age_ids, seg_ids, pos_ids, attention_mask, labels)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"🟦 Fine-tune Epoch {epoch} VAL loss = {avg_val_loss:.4f}")

    # ===== SAVE BEST =====
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        wait = 0
        torch.save(model.state_dict(), "behrt/result/best_val_model_finetune.pt")
        print(f"🔥 BEST FINE-TUNE SAVED — {best_val_loss:.4f}")
    else:
        wait += 1
        if wait >= patience:
            print("⛔ STOP — NO MORE IMPROVEMENT")
            break

# ============================
# 6) FINAL TEST
# ============================
model.load_state_dict(torch.load("behrt/result/best_val_model_finetune.pt"))
model.eval()

test_loss = 0
with torch.no_grad():
    for batch in test_loader:
        visits, age_ids, seg_ids, pos_ids, attention_mask, labels = batch
        visits, age_ids, seg_ids, pos_ids, attention_mask, labels = \
            visits.to(device), age_ids.to(device), seg_ids.to(device), pos_ids.to(device), attention_mask.to(device), labels.to(device)
        loss, _ = model(visits, age_ids, seg_ids, pos_ids, attention_mask, labels)
        test_loss += loss.item()

print(f"\n🟨 FINAL TEST LOSS (FINE-TUNE) = {test_loss/ len(test_loader):.4f}")

✔ Loaded mimic fine-tune datasets


t_total value of -1 results in schedule not being applied


🔹 Loading pretrained weights from synthetic ...
✔ Model loaded on cuda
✔ Finished freezing early layers


/usr/local/lib/python3.11/dist-packages/pytorch_pretrained_bert/optimization.py:275: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha = 1) (Triggered internally at /pytorch/torch/csrc/utils/python_arg_parser.cpp:1661.)
  next_m.mul_(beta1).add_(1 - beta1, grad)



🟩 Fine-tune Epoch 0 TRAIN loss = 0.0217
🟦 Fine-tune Epoch 0 VAL loss = 0.0210
🔥 BEST FINE-TUNE SAVED — 0.0210

🟩 Fine-tune Epoch 1 TRAIN loss = 0.0210
🟦 Fine-tune Epoch 1 VAL loss = 0.0208
🔥 BEST FINE-TUNE SAVED — 0.0208

🟩 Fine-tune Epoch 2 TRAIN loss = 0.0208
🟦 Fine-tune Epoch 2 VAL loss = 0.0208
🔥 BEST FINE-TUNE SAVED — 0.0208

🟩 Fine-tune Epoch 3 TRAIN loss = 0.0207
🟦 Fine-tune Epoch 3 VAL loss = 0.0207
🔥 BEST FINE-TUNE SAVED — 0.0207

🟩 Fine-tune Epoch 4 TRAIN loss = 0.0206
🟦 Fine-tune Epoch 4 VAL loss = 0.0207

🟩 Fine-tune Epoch 5 TRAIN loss = 0.0206
🟦 Fine-tune Epoch 5 VAL loss = 0.0208

🟩 Fine-tune Epoch 6 TRAIN loss = 0.0206
🟦 Fine-tune Epoch 6 VAL loss = 0.0208

🟩 Fine-tune Epoch 7 TRAIN loss = 0.0206
🟦 Fine-tune Epoch 7 VAL loss = 0.0208

🟩 Fine-tune Epoch 8 TRAIN loss = 0.0205
🟦 Fine-tune Epoch 8 VAL loss = 0.0208
⛔ STOP — NO MORE IMPROVEMENT

🟨 FINAL TEST LOSS (FINE-TUNE) = 0.0198


In [8]:
import torch
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

from behrt.data_synth_loader import SyntheticBEHRTDataset
from behrt.data_mimic_loader import RealBEHRTDataset
from behrt.model import BertForMultiLabelPrediction, BertConfig

# =========================
# 1. CẤU HÌNH MODEL
# =========================
config = {
    'vocab_size': 2871,
    'hidden_size': 288,
    'seg_vocab_size': 34,
    'age_vocab_size': 2,
    'max_position_embedding': 1500,
    'hidden_dropout_prob': 0.1,
    'num_hidden_layers': 4,
    'num_attention_heads': 8,
    'attention_probs_dropout_prob': 0.1,
    'intermediate_size': 512,
    'hidden_act': 'gelu',
    'initializer_range': 0.02,
}

feature_dict = {
    'word': True,
    'seg': True,
    'age': False,
    'position': True
}

model_config = BertConfig(config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def evaluate(model_path, dataset, desc=""):
    print(f"\n🔹 Evaluating: {desc} ({model_path})")

    model = BertForMultiLabelPrediction(
        model_config,
        num_labels=config['vocab_size'],
        feature_dict=feature_dict
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    all_y_true = []
    all_y_pred = []

    loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=False)

    with torch.no_grad():
        for batch in loader:
            visits, age_ids, seg_ids, pos_ids, attention_mask, labels = batch

            visits = visits.to(device)
            age_ids = age_ids.to(device)
            seg_ids = seg_ids.to(device)
            pos_ids = pos_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            logits = model(visits, age_ids, seg_ids, pos_ids, attention_mask)
            pred_prob = torch.sigmoid(logits).cpu().numpy()
            true = labels.cpu().numpy()

            all_y_true.append(true)
            all_y_pred.append(pred_prob)

    all_y_true = np.concatenate(all_y_true, axis=0)
    all_y_pred = np.concatenate(all_y_pred, axis=0)

    # ===== AUC micro (flatten) =====
    auc = roc_auc_score(all_y_true.reshape(-1), all_y_pred.reshape(-1))

    # ===== Threshold & classification =====
    threshold = 0.2
    all_y_pred_bin = (all_y_pred > threshold).astype(int)

    f1 = f1_score(all_y_true, all_y_pred_bin, average="micro")
    p  = precision_score(all_y_true, all_y_pred_bin, average="micro")
    r  = recall_score(all_y_true, all_y_pred_bin, average="micro")

    print(f"AUC       = {auc:.4f}")
    print(f"F1        = {f1:.4f}")
    print(f"Precision = {p:.4f}")
    print(f"Recall    = {r:.4f}")

    return auc, f1, p, r


# ================================
# 2. LOAD TEST DATASETS
# ================================
synth_test = SyntheticBEHRTDataset("/kaggle/input/behrt-ft-cp/behrt_compare/synth_test.npz", max_len=512)
real_test  = RealBEHRTDataset("/kaggle/input/behrt-ft-cp/behrt_compare/mimic_test.npz",  max_len=512)

print("====================================")
print("         EVALUATION RESULTS         ")
print("====================================")

# đường dẫn — chỉnh lại đúng path của bạn
PATH_SYNTH      = "/kaggle/input/behrt-ft-cp/behrt_compare/best_val_model.pt"
PATH_REAL       = "/kaggle/input/behrt-ft-cp/behrt_compare/best_val_model_real.pt"
PATH_FINETUNED  = "/kaggle/input/behrt-finetune/best_val_model_finetune.pt"

# 1) Model train synthetic ONLY, test trên real
auc_s_r, f1_s_r, p_s_r, r_s_r = evaluate(PATH_SYNTH, real_test, desc="Synthetic-only model (test on REAL)")

# 2) Model train real from scratch
auc_r, f1_r, p_r, r_r = evaluate(PATH_REAL, real_test, desc="Real-only model (MIMIC from scratch)")

# 3) Model pretrain synthetic → fine-tune real
auc_ft, f1_ft, p_ft, r_ft = evaluate(PATH_FINETUNED, real_test, desc="Finetuned model (Synthetic → Real)")


print("\n====================================")
print("           COMPARISON (REAL TEST)   ")
print("====================================")
print(f"AUC   : SYNTH_ONLY = {auc_s_r:.4f} | REAL_ONLY = {auc_r:.4f} | FINETUNE = {auc_ft:.4f}")
print(f"F1    : SYNTH_ONLY = {f1_s_r:.4f} | REAL_ONLY = {f1_r:.4f} | FINETUNE = {f1_ft:.4f}")
print(f"P     : SYNTH_ONLY = {p_s_r:.4f} | REAL_ONLY = {p_r:.4f} | FINETUNE = {p_ft:.4f}")
print(f"R     : SYNTH_ONLY = {r_s_r:.4f} | REAL_ONLY = {r_r:.4f} | FINETUNE = {r_ft:.4f}")


         EVALUATION RESULTS         

🔹 Evaluating: Synthetic-only model (test on REAL) (/kaggle/input/behrt-ft-cp/behrt_compare/best_val_model.pt)
AUC       = 0.8607
F1        = 0.0948
Precision = 0.2361
Recall    = 0.0593

🔹 Evaluating: Real-only model (MIMIC from scratch) (/kaggle/input/behrt-ft-cp/behrt_compare/best_val_model_real.pt)
AUC       = 0.8961
F1        = 0.1545
Precision = 0.3080
Recall    = 0.1031

🔹 Evaluating: Finetuned model (Synthetic → Real) (/kaggle/input/behrt-finetune/best_val_model_finetune.pt)
AUC       = 0.8983
F1        = 0.1545
Precision = 0.3080
Recall    = 0.1031

           COMPARISON (REAL TEST)   
AUC   : SYNTH_ONLY = 0.8607 | REAL_ONLY = 0.8961 | FINETUNE = 0.8983
F1    : SYNTH_ONLY = 0.0948 | REAL_ONLY = 0.1545 | FINETUNE = 0.1545
P     : SYNTH_ONLY = 0.2361 | REAL_ONLY = 0.3080 | FINETUNE = 0.3080
R     : SYNTH_ONLY = 0.0593 | REAL_ONLY = 0.1031 | FINETUNE = 0.1031
